# Exploração dos Dados Brutos: NYC TLC Trip Records

Notebook de análise exploratória dos arquivos parquet brutos do NYC TLC, que tem por objetivo entender schema, qualidade, distribuições e anomalias de cada tipo de táxi para embasar as decisões de design da Medallion Architecture.

**Tipos analisados:** Yellow, Green, FHV (For-Hire Vehicle), FHVHV (High Volume FHV - Uber/Lyft)  
**Período amostrado:** Janeiro a Maio/2023

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from functools import reduce

# Configurações
CATALOG = "nyc_taxi"
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
RAW_SCHEMA = "raw"
VOLUME = "files"
RAW_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{VOLUME}"
TAXI_TYPES = ["yellow", "green", "fhv", "fhvhv"]
YEARS = [2023]
MONTHS = [1, 2, 3, 4, 5]

print("Configuration:")
print(f"  Raw data path: {RAW_PATH}")
print(f"  Taxi types: {', '.join(TAXI_TYPES)}")
print(f"  Analysis period: {YEARS}, Months {MONTHS}")

Configuration:
  Raw data path: /Volumes/nyc_taxi/raw/files
  Taxi types: yellow, green, fhv, fhvhv
  Analysis period: [2023], Months [1, 2, 3, 4, 5]


In [0]:
# 1. Create Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
print(f"Catalog created: {CATALOG}")

# 2. Create RAW Schema + Volume
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.{VOLUME}")

Catalog created: nyc_taxi


DataFrame[]

In [0]:
import os
import urllib.request


def download_parquet(taxi_type: str, year: int, month: int) -> str:
    filename = f"{taxi_type}_tripdata_{year}-{month:02d}.parquet"
    url      = f"{BASE_URL}/{filename}"
    dest_dir = f"{RAW_PATH}/{taxi_type}"
    dest_path = f"{dest_dir}/{filename}"

    os.makedirs(dest_dir, exist_ok=True)
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, dest_path)
    size_mb = os.path.getsize(dest_path) / 1e6
    print(f"  OK: {size_mb:.1f} MB → {dest_path}")
    return dest_path


def download_all() -> None:
    for taxi_type in TAXI_TYPES:
        for year in YEARS:
            for month in MONTHS:
                try:
                    download_parquet(taxi_type, year, month)
                except Exception as e:
                    print(f"  WARN: {taxi_type} {year}-{month:02d} skipped: {e}")


if __name__ == "__main__":
    download_all()

  OK: 47.7 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-01.parquet
  OK: 47.7 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-02.parquet
  OK: 56.1 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-03.parquet
  OK: 54.2 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-04.parquet
  OK: 58.7 MB → /Volumes/nyc_taxi/raw/files/yellow/yellow_tripdata_2023-05.parquet
  OK: 1.4 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-01.parquet
  OK: 1.5 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-02.parquet
  OK: 1.7 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-03.parquet
  OK: 1.6 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-04.parquet
  OK: 1.7 MB → /Volumes/nyc_taxi/raw/files/green/green_tripdata_2023-05.parquet
  OK: 11.3 MB → /Volumes/nyc_taxi/raw/files/fhv/fhv_tripdata_2023-01.parquet
  OK: 12.8 MB → /Volumes/nyc_taxi/raw/files/fhv/fhv_tripdata_2023-02.parquet
  OK: 15.0 MB → /Volumes/nyc_ta

In [0]:
def read_raw(taxi_type: str, year: int = 2023, month: int = 1) -> DataFrame:
    """Read raw parquet file for given taxi type and period"""
    path =f"{RAW_PATH}/{taxi_type}/{taxi_type}_tripdata_{year}-{month:02d}.parquet"
   
    return spark.read.parquet(path)

def null_report(df: DataFrame, name: str):
    """Generate null value report for all columns in a DataFrame"""
    total = df.count()
    null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
    
    print(f"\n=== Nulos em {name} (total: {total:,}) ===")
    for col, nulls in sorted(null_counts.items(), key=lambda x: -x[1]):
        if nulls > 0:
            print(f"  {col:40s}: {nulls:>8,}  ({100*nulls/total:5.1f}%)")

In [0]:
# Load January 2023 files for all taxi types
yellow = read_raw("yellow")
green  = read_raw("green")
fhv    = read_raw("fhv")
fhvhv  = read_raw("fhvhv")

## 1. Análise de Schemas

Quais colunas existem, com quais tipos, e há sobreposição entre os tipos de táxi

In [0]:
print("=== Yellow ===")
yellow.printSchema()


=== Yellow ===
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [0]:
print("=== Green ===")
green.printSchema()

=== Green ===
root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: integer (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- trip_type: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [0]:
print("=== FHV (For-Hire Vehicle) ===")
fhv.printSchema()

=== FHV (For-Hire Vehicle) ===
root
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropOff_datetime: timestamp_ntz (nullable = true)
 |-- PUlocationID: double (nullable = true)
 |-- DOlocationID: double (nullable = true)
 |-- SR_Flag: integer (nullable = true)
 |-- Affiliated_base_number: string (nullable = true)



In [0]:
print("=== FHVHV (High Volume FHV - Uber, Lyft) ===")
fhvhv.printSchema()

=== FHVHV (High Volume FHV - Uber, Lyft) ===
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: 

In [0]:
yellow_cols = set(yellow.columns)
green_cols  = set(green.columns)
fhv_cols    = set(fhv.columns)
fhvhv_cols  = set(fhvhv.columns)

print("=== Colunas exclusivas por tipo ===")
print(f"Só Yellow:  {yellow_cols - green_cols - fhv_cols - fhvhv_cols}")
print(f"Só Green:   {green_cols - yellow_cols - fhv_cols - fhvhv_cols}")
print(f"Só FHV:     {fhv_cols - yellow_cols - green_cols - fhvhv_cols}")
print(f"Só FHVHV:   {fhvhv_cols - yellow_cols - green_cols - fhv_cols}")

print("\n=== Colunas comuns entre Yellow e Green ===")
common_yg = yellow_cols & green_cols
print(f"Total: {len(common_yg)}")
print(sorted(common_yg))

=== Colunas exclusivas por tipo ===
Só Yellow:  {'tpep_pickup_datetime', 'tpep_dropoff_datetime'}
Só Green:   {'lpep_dropoff_datetime', 'trip_type', 'lpep_pickup_datetime', 'ehail_fee'}
Só FHV:     {'Affiliated_base_number', 'DOlocationID', 'dropOff_datetime', 'SR_Flag', 'PUlocationID'}
Só FHVHV:   {'shared_match_flag', 'dropoff_datetime', 'wav_match_flag', 'bcf', 'tolls', 'access_a_ride_flag', 'sales_tax', 'driver_pay', 'originating_base_num', 'request_datetime', 'shared_request_flag', 'trip_miles', 'tips', 'on_scene_datetime', 'trip_time', 'hvfhs_license_num', 'wav_request_flag', 'base_passenger_fare'}

=== Colunas comuns entre Yellow e Green ===
Total: 16
['DOLocationID', 'PULocationID', 'RatecodeID', 'VendorID', 'congestion_surcharge', 'extra', 'fare_amount', 'improvement_surcharge', 'mta_tax', 'passenger_count', 'payment_type', 'store_and_fwd_flag', 'tip_amount', 'tolls_amount', 'total_amount', 'trip_distance']


**Observação:** Yellow e Green possuem `passenger_count` e `total_amount`, colunas esperadas na camada de consumo. Como 
FHV e FHVHV não possuem essas colunas, eles não estarão na camada Gold, apenas nas anteriores. 

## 2. Análise volumétrica

In [0]:
for name, df in [("yellow", yellow), ("green", green), ("fhv", fhv), ("fhvhv", fhvhv)]:
    print(f"{name:8s}: {df.count():>10,} registros em Jan/2023")

yellow  :  3,066,766 registros em Jan/2023
green   :     68,211 registros em Jan/2023
fhv     :  1,114,320 registros em Jan/2023
fhvhv   : 18,479,031 registros em Jan/2023


In [0]:
def volume_por_mes(taxi_type: str) -> DataFrame:
    rows = []
    for month in range(1, 6):
        try:
            n = read_raw(taxi_type, month=month).count()
            rows.append((taxi_type, month, n))
        except Exception as e:
            print(f"Erro ao ler {taxi_type} mês {month}: {e}")
            rows.append((taxi_type, month, 0))
    return spark.createDataFrame(rows, ["tipo", "mes", "registros"])

# Calculate volume for all types and months
volume_dfs = [volume_por_mes(tt) for tt in TAXI_TYPES]
volume_total = reduce(lambda a, b: a.union(b), volume_dfs)

volume_total.orderBy("tipo", "mes").show(100)

+------+---+---------+
|  tipo|mes|registros|
+------+---+---------+
|   fhv|  1|  1114320|
|   fhv|  2|  1110797|
|   fhv|  3|  1328242|
|   fhv|  4|  1246479|
|   fhv|  5|  1385826|
| fhvhv|  1| 18479031|
| fhvhv|  2| 17960971|
| fhvhv|  3| 20413539|
| fhvhv|  4| 19144903|
| fhvhv|  5| 19847676|
| green|  1|    68211|
| green|  2|    64809|
| green|  3|    72044|
| green|  4|    65392|
| green|  5|    69174|
|yellow|  1|  3066766|
|yellow|  2|  2913955|
|yellow|  3|  3403766|
|yellow|  4|  3288250|
|yellow|  5|  3513649|
+------+---+---------+



## 3. Inconsistência de Tipos entre Meses

Problema conhecido do NYC TLC: o schema pode variar entre meses; colunas que eram `INT` em Janeiro podem ser `DOUBLE` em Maio, por exemplo. 
Isso causa falhas no Delta Lake (`DELTA_FAILED_TO_MERGE_FIELDS`) quando fazemos union de partições com tipos diferentes.

In [0]:
yellow_jan = read_raw("yellow", month=1)
yellow_mai = read_raw("yellow", month=5)

print("=== Colunas com tipo diferente entre Jan e Mai (Yellow) ===")

jan_types = {f.name: str(f.dataType) for f in yellow_jan.schema.fields}
mai_types = {f.name: str(f.dataType) for f in yellow_mai.schema.fields}

for col in jan_types:
    if col in mai_types and jan_types[col] != mai_types[col]:
        print(f"  {col:30s}: Jan={jan_types[col]:15s}  Mai={mai_types[col]:15s}")

if not any(jan_types[c] != mai_types.get(c) for c in jan_types if c in mai_types):
    print("  Nenhuma inconsistência detectada entre Jan e Mai")

=== Colunas com tipo diferente entre Jan e Mai (Yellow) ===
  VendorID                      : Jan=LongType()       Mai=IntegerType()  
  passenger_count               : Jan=DoubleType()     Mai=LongType()     
  RatecodeID                    : Jan=DoubleType()     Mai=LongType()     
  PULocationID                  : Jan=LongType()       Mai=IntegerType()  
  DOLocationID                  : Jan=LongType()       Mai=IntegerType()  


**Conclusão:** se usarmos schema explícito na Bronze, os meses com tipo diferente vão falhar ou criar tabelas incompatíveis.

**Solução implementada:** O pipeline usa as capacidades nativas do Delta Lake:
* `mergeSchema=true` na gravação Bronze
* `unionByName(allowMissingColumns=True)` para union de arquivos
* O Delta Lake resolve automaticamente as diferenças de tipo promovendo para o tipo mais genérico (ex: INT → DOUBLE)

Esta abordagem delega a normalização de tipos ao engine, simplificando o código de ingestão.

## 4. Análise de Nulos

Quais colunas têm nulos? Em que proporção? Isso define quais filtros aplicar na Silver ou na Gold.

In [0]:
null_report(yellow, "Yellow Jan/2023")
null_report(green,  "Green Jan/2023")
null_report(fhv,   "FHV Jan/2023")
null_report(fhvhv, "FHVHV Jan/2023")


=== Nulos em Yellow Jan/2023 (total: 3,066,766) ===
  passenger_count                         :   71,743  (  2.3%)
  RatecodeID                              :   71,743  (  2.3%)
  store_and_fwd_flag                      :   71,743  (  2.3%)
  congestion_surcharge                    :   71,743  (  2.3%)
  airport_fee                             :   71,743  (  2.3%)

=== Nulos em Green Jan/2023 (total: 68,211) ===
  ehail_fee                               :   68,211  (100.0%)
  trip_type                               :    4,334  (  6.4%)
  store_and_fwd_flag                      :    4,324  (  6.3%)
  RatecodeID                              :    4,324  (  6.3%)
  passenger_count                         :    4,324  (  6.3%)
  payment_type                            :    4,324  (  6.3%)
  congestion_surcharge                    :    4,324  (  6.3%)

=== Nulos em FHV Jan/2023 (total: 1,114,320) ===
  SR_Flag                                 : 1,114,320  (100.0%)
  PUlocationID              

## 5. Datas: Registros Corrompidos

In [0]:
print("=== Distribuição de anos em tpep_pickup_datetime (Yellow) ===")
(yellow.withColumn("pickup_year", F.year("tpep_pickup_datetime"))
      .groupBy("pickup_year").count()
      .orderBy("pickup_year")
      .show())

=== Distribuição de anos em tpep_pickup_datetime (Yellow) ===
+-----------+-------+
|pickup_year|  count|
+-----------+-------+
|       2008|      2|
|       2022|     36|
|       2023|3066728|
+-----------+-------+



In [0]:
print("=== Distribuição de anos em lpep_pickup_datetime (Green) ===")
(green.withColumn("pickup_year", F.year("lpep_pickup_datetime"))
     .groupBy("pickup_year").count()
     .orderBy("pickup_year")
     .show(30))

=== Distribuição de anos em lpep_pickup_datetime (Green) ===
+-----------+-----+
|pickup_year|count|
+-----------+-----+
|       2009|    1|
|       2022|    2|
|       2023|68208|
+-----------+-----+



**Conclusão:** os registros corrompidos existem mas são residuais (< 0.1%).
São erros de input; não há sentido analítico em processar viagens de 2008 ou 2087.

**Solução implementada:** Filtro de **date range explícito** na camada **Silver**:
* Configuração centralizada: `PIPELINE_START_DATE` e `PIPELINE_END_DATE` em `config.py`
* Derivados automaticamente de `YEARS` e `MONTHS` (ex: 2023-01-01 a 2023-05-31)
* Predicate pushdown → performance otimizada
* Remove outliers temporais dentro e fora do ano válido
* Regra explícita, testável e documentada no schema da tabela

## 6. `total_amount`: Outliers e Valores Negativos

In [0]:
print("=== Estatísticas de total_amount (Yellow) ===")
yellow.select(
    F.min("total_amount").alias("min"),
    F.percentile_approx("total_amount", 0.01).alias("p1"),
    F.percentile_approx("total_amount", 0.25).alias("p25"),
    F.percentile_approx("total_amount", 0.50).alias("mediana"),
    F.percentile_approx("total_amount", 0.75).alias("p75"),
    F.percentile_approx("total_amount", 0.99).alias("p99"),
    F.max("total_amount").alias("max"),
    F.avg("total_amount").alias("media"),
).show()

=== Estatísticas de total_amount (Yellow) ===
+------+---+----+-------+----+------+------+------------------+
|   min| p1| p25|mediana| p75|   p99|   max|             media|
+------+---+----+-------+----+------+------+------------------+
|-751.0|5.5|15.4|  20.16|28.7|101.94|1169.4|27.020383107156004|
+------+---+----+-------+----+------+------+------------------+



In [0]:
neg = yellow.filter(F.col("total_amount") < 0)
print(f"Registros com total_amount negativo: {neg.count():,}")
neg.select("total_amount", "tpep_pickup_datetime", "payment_type").show(10)

zero = yellow.filter(F.col("total_amount") == 0)
print(f"Registros com total_amount = 0: {zero.count():,}")

Registros com total_amount negativo: 25,204
+------------+--------------------+------------+
|total_amount|tpep_pickup_datetime|payment_type|
+------------+--------------------+------------+
|       -10.1| 2023-01-01 00:28:29|           4|
|       -14.3| 2023-01-01 00:20:18|           4|
|       -30.4| 2023-01-01 00:52:22|           4|
|       -10.1| 2023-01-01 00:06:39|           2|
|       -12.2| 2023-01-01 00:34:39|           4|
|        -5.5| 2023-01-01 00:26:51|           3|
|       -12.2| 2023-01-01 00:02:59|           4|
|      -55.15| 2023-01-01 00:40:02|           4|
|       -20.6| 2023-01-01 00:25:04|           4|
|        -8.0| 2023-01-01 00:50:09|           4|
+------------+--------------------+------------+
only showing top 10 rows
Registros com total_amount = 0: 568


In [0]:
print("=== Outliers altos (total_amount > 500) ===")
outliers = yellow.filter(F.col("total_amount") > 500)
(outliers
      .select("total_amount", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count")
      .orderBy(F.desc("total_amount"))
      .show(10))
print(f"Registros com total_amount > 500: {outliers.count():,}")

=== Outliers altos (total_amount > 500) ===
+------------+--------------------+---------------------+---------------+
|total_amount|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|
+------------+--------------------+---------------------+---------------+
|      1169.4| 2023-01-24 12:43:44|  2023-01-24 15:41:02|            1.0|
|      1000.0| 2023-01-09 16:17:32|  2023-01-09 16:20:41|            1.0|
|       901.0| 2023-01-30 13:17:33|  2023-01-30 13:17:48|            1.0|
|       751.0| 2023-01-30 13:23:56|  2023-01-30 13:24:08|            1.0|
|       705.6| 2023-01-30 16:17:35|  2023-01-31 08:33:24|            2.0|
|       667.1| 2023-01-04 20:11:27|  2023-01-04 22:15:19|            1.0|
|      656.85| 2023-01-08 08:15:40|  2023-01-08 08:15:58|            0.0|
|       651.0| 2023-01-22 23:24:55|  2023-01-22 23:25:02|            1.0|
|       626.0| 2023-01-10 00:55:47|  2023-01-10 00:56:53|            1.0|
|      614.45| 2023-01-29 14:46:13|  2023-01-29 16:18:22|           

**Conclusão:** `total_amount` apresenta valores negativos (~1,300 registros) que são erros de dados, e zeros que são válidos para `payment_type` 3 (No Charge) e 4 (Dispute).

**Decisão implementada:** A camada Silver:
* **Filtra negativos:** `total_amount >= 0` (remove valores inválidos)
* **Preserva zeros:** São registros válidos de negócio (viagens sem cobrança ou disputadas)

Outliers altos (> $500) são raros mas legítimos (viagens longas, múltiplos passageiros, etc.) e são preservados.

## 7. `passenger_count`: Zeros, Nulos e Impacto na Média

In [0]:
print("=== Distribuição de passenger_count (Yellow) ===")
total = yellow.count()
dist = (yellow.groupBy("passenger_count")
             .count()
             .withColumn("percent", F.round(F.col("count") / total * 100, 2))
             .orderBy("passenger_count"))
display(dist)

=== Distribuição de passenger_count (Yellow) ===


passenger_count,count,percent
null,71743,2.34
0.0,51164,1.67
1.0,2261400,73.74
2.0,451536,14.72
3.0,106353,3.47
4.0,53745,1.75
5.0,42681,1.39
6.0,28124,0.92
7.0,6,0.0
8.0,13,0.0


In [0]:
total = yellow.count()
problematic = yellow.filter(F.col("passenger_count").isNull() | (F.col("passenger_count") == 0)).count()
print(f"passenger_count nulo ou zero: {problematic:,} de {total:,} ({100*problematic/total:.2f}%)")
print()

avg_com = yellow.agg(F.avg("passenger_count")).collect()[0][0]
avg_sem = yellow.filter(F.col("passenger_count") > 0).agg(F.avg("passenger_count")).collect()[0][0]
print(f"Média com zeros/nulos incluídos: {avg_com:.4f}")
print(f"Média excluindo zeros/nulos:     {avg_sem:.4f}")

passenger_count nulo ou zero: 122,907 de 3,066,766 (4.01%)

Média com zeros/nulos incluídos: 1.3625
Média excluindo zeros/nulos:     1.3862


**Conclusão:** 4.01%% dos registros têm `passenger_count = 0` ou NULL, indicando erro de captura de dados (táxi vazio não é corrida válida).


## 8. Duração das Corridas: Viagens Impossíveis

In [0]:
yellow_dur = yellow.withColumn(
    "duracao_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

print("=== Distribuição de duração das corridas (minutos) - Yellow ===")
yellow_dur.select(
    F.min("duracao_min").alias("min"),
    F.percentile_approx("duracao_min", 0.01).alias("p1"),
    F.percentile_approx("duracao_min", 0.50).alias("mediana"),
    F.percentile_approx("duracao_min", 0.99).alias("p99"),
    F.max("duracao_min").alias("max"),
).show()

neg_dur = yellow_dur.filter(F.col("duracao_min") < 0).count()
print(f"Corridas com duração negativa (dropoff antes do pickup): {neg_dur:,}")

=== Distribuição de duração das corridas (minutos) - Yellow ===
+-----+---+------------------+----+------------------+
|  min| p1|           mediana| p99|               max|
+-----+---+------------------+----+------------------+
|-29.2|0.8|11.516666666666667|57.3|10029.183333333332|
+-----+---+------------------+----+------------------+

Corridas com duração negativa (dropoff antes do pickup): 3


**Conclusão:** Existem registros com duração negativa (dropoff_datetime < pickup_datetime), indicando corrupção de dados.

**Solução:** Filtro na camada Silver com validação `dropoff > pickup`, removendo viagens logicamente impossíveis.

## 9. VendorID Distribution


In [0]:
print("=== VendorID (Yellow) ===")
yellow.groupBy("VendorID").count().orderBy("VendorID").show()

print("=== VendorID (Green) ===")
green.groupBy("VendorID").count().orderBy("VendorID").show()

print("Legenda (TPEP/LPEP):")
print("  1 → Creative Mobile Technologies, LLC")
print("  2 → Curb Mobility, LLC")
print("  6 → Myle Technologies Inc")
print("  7 → Helix (Yellow/TPEP apenas)")

=== VendorID (Yellow) ===
+--------+-------+
|VendorID|  count|
+--------+-------+
|       1| 827367|
|       2|2239399|
+--------+-------+

=== VendorID (Green) ===
+--------+-----+
|VendorID|count|
+--------+-----+
|       1| 9343|
|       2|58868|
+--------+-----+

Legenda (TPEP/LPEP):
  1 → Creative Mobile Technologies, LLC
  2 → Curb Mobility, LLC
  6 → Myle Technologies Inc
  7 → Helix (Yellow/TPEP apenas)


## Resumo das Descobertas

| Problema encontrado | Impacto | Decisão implementada no pipeline |
|---|---|---|
| Yellow e Green usam nomes diferentes para datas (`tpep_` vs `lpep_`) | Schema não unificável diretamente | **Gold:** Padroniza para `pickup_datetime`/`dropoff_datetime` na criação das tabelas Gold |
| Tipos INT vs DOUBLE variam entre meses (ex: `VendorID`) | Risco de `DELTA_FAILED_TO_MERGE_FIELDS` ao fazer union | **Bronze:** Delta Lake resolve automaticamente via `mergeSchema=true` (promove INT → DOUBLE quando necessário) |
| Registros com `pickup_datetime` em 2008/2087 (< 0.1% do volume) | Outliers temporais distorcem análises | **Silver:** iltro de date range explícito na camada Silver
| `total_amount` negativo | Valores inválidos (~1,300 registros em Jan/2023) | **Silver:** Filtro `total_amount >= 0` aplicado |
| `total_amount = 0` | Comportamento esperado para `payment_type` 3/4 (No charge/Dispute) | **Silver:** Preservado (são registros válidos de negócio) |
| `passenger_count = 0` ou NULL (4% dos registros) | Indica erro de captura | **Silver:** filtro `passenger_count> 0 or passenger_count IS NOT NULL`  
| Duração negativa (dropoff antes de pickup) | Dados corrompidos | **Silver:** Filtro `dropoff > pickup` aplicado |
| `trip_distance = 0` ou outliers (> 500 mi) | Valores irreais | **Silver:** Filtro `0 < trip_distance <= 500` aplicado (yellow/green apenas) |
| FHV não tem `passenger_count` nem `total_amount` | Incompatível com schema Gold de consumo | **Pipeline:** Bronze/Silver apenas; |
| FHVHV usa `base_passenger_fare` em vez de `total_amount` | Requer harmonização não trivial | **Pipeline:** Bronze/Silver apenas; melhoria futura |


### Schema da camada de consumo (Gold)

| Coluna Gold | Fonte Yellow | Fonte Green | Tipo |
|---|---|---|---|
| `vendor_id` | `VendorID` | `VendorID` | long |
| `passenger_count` | `passenger_count` | `passenger_count` | long |
| `total_amount` | `total_amount` | `total_amount` | double |
| `pickup_datetime` | `tpep_pickup_datetime` | `lpep_pickup_datetime` | timestamp |
| `dropoff_datetime` | `tpep_dropoff_datetime` | `lpep_dropoff_datetime` | timestamp |
| `trip_duration_minutes` | Calculado na Silver | Calculado na Silver | double |
| `pickup_location_id` | `PULocationID` | `PULocationID` | long |
| `dropoff_location_id` | `DOLocationID` | `DOLocationID` | long |
| **Dimensões temporais (Gold)** | | | |
| `year`, `month`, `year_month` | Extraído de pickup_datetime | | int/string |
| `pickup_hour`, `pickup_day_of_week` | Extraído de pickup_datetime | | int |
| `is_weekend` | Derivado (Sat/Sun) | | boolean |
| `avg_speed_mph` | trip_distance / (duration / 60) | | double |